In [181]:
%pip install -U langchain langchain-community langchain-text-splitters langchain-chroma langchain-huggingface chromadb sentence-transformers transformers accelerate torch ipykernel jupyter faiss-cpu

Note: you may need to restart the kernel to use updated packages.



THE FOLLOWING FEW CELLS WILL:

load merged_corpus.jsonl

convert rows into LangChain Documents

chunk them

embed them

save them into Chroma

In [299]:
#imports
import json
import re
from pathlib import Path
import os

# LangChain 
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Models
from sentence_transformers import CrossEncoder
from transformers import pipeline

In [311]:
#paths and config
CORPUS_PATH = Path("merged_corpus.jsonl")
FAISS_DIR = "faiss_store"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

INITIAL_RETRIEVAL_K = 12  
FINAL_K = 5                
MAX_NEW_TOKENS = 250
MIN_OVERLAP_SCORE = 2 

In [312]:
def load_corpus(file_path):
    docs = []

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)

            docs.append(
                Document(
                    page_content=item["text"],
                    metadata={
                        "title": item.get("title", ""),
                        "source": item.get("source", "unknown")
                    }
                )
            )

    return docs


docs = load_corpus(CORPUS_PATH)
print(f"Loaded {len(docs)} documents")

Loaded 58 documents


In [313]:
#chunk the documents
# RecursiveCharacterTextSplitter - LangChain for text splitting.

def chunk_documents(docs, chunk_size=500):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=100
    )

    return splitter.split_documents(docs)


chunks = chunk_documents(docs, 500)
chunks = split_documents(docs, 500, 100)

In [314]:
#create embeddings

embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

vectorstore = FAISS.from_documents(chunks, embedding)
print("Vector store created")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector store created


In [315]:
def normalize_text(text):
    return re.sub(r"\s+", " ", text.strip().lower())

def deduplicate_docs(docs):
    seen = set()
    unique_docs = []

    for doc in docs:
        title = normalize_text(doc.metadata.get("title", ""))
        content = normalize_text(doc.page_content)

        key = (title, content[:300])

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    return unique_docs

def keyword_overlap_score(query, text):
    query_words = set(re.findall(r"\w+", query.lower()))
    text_words = set(re.findall(r"\w+", text.lower()))
    return len(query_words & text_words)

def recipe_bonus(doc):
    title = doc.metadata.get("title", "").lower()
    text = doc.page_content.lower()

    bonus = 0

    if "cookbook:" in title:
        bonus += 2
    if "ingredients" in text:
        bonus += 1
    if "procedure" in text or "method" in text or "directions" in text:
        bonus += 1

    return bonus

def expand_query(query):
    expansions = [query]

    q = query.lower().strip()

    if q.startswith("what is "):
        term = q.replace("what is ", "").strip()
        expansions.extend([
            f"{term} ingredients",
            f"{term} origin",
            f"{term} recipe"
        ])

    elif q.startswith("recipe for "):
        term = q.replace("recipe for ", "").strip()
        expansions.extend([
            f"{term} ingredients",
            f"{term} method",
            f"{term} cookbook"
        ])

    elif "ingredients" in q:
        expansions.extend([
            query.replace("ingredients", "recipe"),
            query.replace("ingredients", "method")
        ])

    seen = set()
    final_expansions = []

    for item in expansions:
        if item not in seen:
            seen.add(item)
            final_expansions.append(item)

    return final_expansions

def rerank_docs(query, docs):
    scored = []

    for doc in docs:
        overlap = keyword_overlap_score(query, doc.page_content)
        bonus = recipe_bonus(doc)

        final_score = overlap + bonus

        doc.metadata["overlap"] = overlap
        doc.metadata["rerank_score"] = final_score

        scored.append((final_score, doc))

    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored]
    return answer
    
def retrieve_docs(query, vectorstore, k=INITIAL_RETRIEVAL_K, final_k=FINAL_K):
    expanded_queries = expand_query(query)
    all_docs = []

    for q in expanded_queries:
        docs = vectorstore.similarity_search(q, k=k)
        all_docs.extend(docs)

    all_docs = deduplicate_docs(all_docs)
    all_docs = rerank_docs(query, all_docs)

    return all_docs[:final_k]


In [291]:
def extract_ingredients(contexts):
    text = " ".join(contexts).lower()

    if "ingredients such as" in text:
        part = text.split("ingredients such as")[1]
        part = part.split(".")[0]

        return part.strip()

    return None

def is_recipe_query(query):
    keywords = ["recipe", "how to make", "how is", "steps", "method"]
    return any(k in query.lower() for k in keywords)

def extract_recipe(query, contexts):
    query_terms = query.lower().split()

    filtered = [
        c for c in contexts
        if any(term in c.lower() for term in query_terms)
    ]

    if not filtered:
        return None

    text = filtered[0]

    steps = []

    for line in text.split("\n"):
        if any(word in line.lower() for word in ["mix", "add", "bake", "boil", "pour"]):
            steps.append(line.strip())

    return "\n".join(steps) if steps else None

In [316]:
def build_prompt(query, contexts):
    context_text = "\n\n".join(contexts)

    return f"""
You are a culinary question-answering assistant.

Answer using ONLY the context below.
If the answer is not in the context, say:
"I don't know based on the provided context."

If the user asks for a recipe, give:
- Recipe name
- Ingredients
- Method

Keep the answer concise and complete.
Do not invent information.


Context:
{context_text}

Question:
{query}

Answer:
"""

In [317]:
generator = pipeline(
    "text-generation",
    model=GENERATION_MODEL
)

def generate_answer(prompt):
    result = generator(
        prompt,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        truncation=True,
        return_full_text=False
    )
    return result[0]["generated_text"]

def clean_answer(answer):
    answer = answer.strip()
    answer = answer.replace("\n", " ")
    answer = " ".join(answer.split())

    bad_starts = [
        "based on the context provided,",
        "according to the context,",
        "from the context provided,"
    ]

    lower_answer = answer.lower()
    for phrase in bad_starts:
        if lower_answer.startswith(phrase):
            answer = answer[len(phrase):].strip(" ,")
            break

    return answer

def enough_query_overlap(query, docs, min_total_overlap=2):
    total_overlap = 0
    for doc in docs:
        total_overlap += keyword_overlap_score(query, doc.page_content)
    return total_overlap >= min_total_overlap

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [318]:
def rag_pipeline(query, return_docs=False):
    retrieved_docs = retrieve_docs(query, vectorstore)

    if not retrieved_docs or not enough_query_overlap(query, retrieved_docs):
        if return_docs:
            return "I don't know based on the provided context.", retrieved_docs
        return "I don't know based on the provided context."

    contexts = [doc.page_content for doc in retrieved_docs]
    prompt = build_prompt(query, contexts)

    raw_answer = generate_answer(prompt)
    final_answer = clean_answer(raw_answer)

    if return_docs:
        return final_answer, retrieved_docs

    return final_answer

In [319]:
def simple_f1(pred, gold):
    pred_tokens = pred.lower().split()
    gold_tokens = gold.lower().split()

    common = set(pred_tokens) & set(gold_tokens)

    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens) if pred_tokens else 0
    recall = len(common) / len(gold_tokens) if gold_tokens else 0

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

In [320]:
query = "recipe for baklava"

answer, retrieved_docs = rag_pipeline(query, return_docs=True)

print("ANSWER:\n")
print(answer)

print("\nRETRIEVED CHUNKS:\n")
for i, doc in enumerate(retrieved_docs, 1):
    score = doc.metadata.get("rerank_score", "N/A")
    overlap = doc.metadata.get("overlap", "N/A")
    title = doc.metadata.get("title", "Untitled")
    print(f"{i}. score={score} | overlap={overlap} | title={title}")

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


ANSWER:

Baklava Recipe Name: Baklava Ingredients: - 1 lb (450 g) nuts (pistachios or walnuts) - 1 tsp (5 ml) ground cinnamon - 1 lb (450 g) phyllo dough - ½ cup (120 ml) butter - 1 tsp (5 ml) vanilla extract - ⅓ cup (80 ml) honey - ⅔ cup (160 ml) white sugar - ¾ cup (180 ml) water Method: 1. On the 6th sheet, sprinkle evenly with nut mixture. 2. Continue until all nut mixture is used and last 6 buttered sheets of Phyllo form top crust. 3. Use a sharp knife to cut into diamond shapes. 4. Bake at 300 °F (150 °C) for about 1 ½ hours or until lightly brown. 5. Pour cooled syrup over hot pastry. Enjoy!

RETRIEVED CHUNKS:

1. score=5 | overlap=2 | title=Cookbook:Baklava II
2. score=5 | overlap=2 | title=Cookbook:Baklava I
3. score=4 | overlap=2 | title=Cookbook:Baklava II
4. score=4 | overlap=1 | title=Cookbook:Galaktoboureko (Greek Semolina Custard Pastry)
5. score=3 | overlap=1 | title=Cookbook:Baklava II


In [310]:
while True:
    query = input("\nEnter your question (or type 'exit' to quit): ").strip()

    if query.lower() == "exit":
        print("Goodbye.")
        break

    if not query:
        print("Please enter a question.")
        continue

    answer, retrieved_docs = rag_pipeline(query, return_docs=True)

    print("\nANSWER:\n")
    print(answer)

    print("\nRETRIEVED CHUNKS:\n")
    for i, doc in enumerate(retrieved_docs, 1):
        score = doc.metadata.get("rerank_score", "N/A")
        overlap = doc.metadata.get("overlap", "N/A")
        title = doc.metadata.get("title", "Untitled")
        print(f"{i}. score={score} | overlap={overlap} | title={title}")



Enter your question (or type 'exit' to quit):  exit


Goodbye.
